|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 1:</h2>|<h1>The Naive Loop<h1>|
|<h2>Section:</h2>|<h1>Incidents<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: the incident file<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

You finished Part 1. You can build the loop. Now you must see when a loop is
broken from the outside, with only the symptoms.

That is the real job. In production nobody tells you "the position ids are off
by one". A user tells you "the answers got worse on Tuesday".

Each ticket below is a real class of failure in LLM serving. Each ticket gives
you a **symptom** and some **evidence**. Some of the evidence is noise, as in a
real incident. Write four lines for each ticket:

1. **Root cause.** One sentence.
2. **The number that proves it.** Not "it looks like". A computation.
3. **The fix.**
4. **The guard.** A test, an assert or an alert that catches it next time.

Four rules:

- The tickets are **not** in the order of the notebooks. First you must find
  which idea the ticket needs. That is half of the skill.
- At least one ticket is **not a bug**. "Nothing is broken" is a valid answer
  only if a number proves it.
- Write your answer **before** you open the solution. A wrong answer that you
  wrote down teaches you more than a right answer that you read.
- Every ticket has a scratch cell. Most tickets need one computation.

This notebook needs no GPU.

**The on-call colleague.** In Claude Code, type `/incident 1.1` (or any
other ticket number) to work a ticket as a conversation. The colleague has
access to the system. Ask for a log, a measurement or an experiment, and it
answers with what the system shows. When you write your four lines, it tells
you which lines are weak, and it asks a question about each one. It does not
tell you the cause until you ask for the solution.

### The reference sheet

The tickets use these numbers. The GPU numbers are the vendor peaks for dense
bf16. Real code gets 70 to 90% of the bandwidth peak.

| GPU | Memory | Bandwidth | bf16 compute |
|---|---|---|---|
| A100 SXM 80GB | 80 GB | 2,039 GB/s | 312 TFLOP/s |
| H100 SXM 80GB | 80 GB | 3,350 GB/s | 989 TFLOP/s |
| L40S | 48 GB | 864 GB/s | 362 TFLOP/s |
| your card | `./vc info` | `./vc info` | `./vc info` |

| Model | Layers | Attention heads | KV heads | head_dim | hidden | vocab | bf16 weights |
|---|---|---|---|---|---|---|---|
| Qwen3-0.6B | 28 | 16 | 8 | 128 | 1024 | 151,936 | 1.5 GB |
| Qwen3-1.7B | 28 | 16 | 8 | 128 | 2048 | 151,936 | 3.44 GB |
| Llama-3-8B | 32 | 32 | 8 | 128 | 4096 | 128,256 | 16.1 GB |

Two formulas from Part 1:

    KV bytes per token = 2 (K and V) x layers x KV heads x head_dim x bytes per value
    decode floor (s)   = bytes read per step / bandwidth

# Ticket 1: the bill

**Severity:** high. **Reported by:** finance.

> The GPU bill for the chat service went up 9x this month. Traffic is flat.

**Evidence**

- Last month the service moved from an old model to Qwen3-1.7B. The team
  changed only the model name.
- The new model has a larger vocabulary than the old one: 151,936 against
  32,000.
- 100% of the responses since the change have exactly 1024 tokens.
  `max_tokens` is 1024.
- A sample response:

      Hello! How can I help you today?<|im_end|>

      Hello! How can I help you today?<|im_end|>

      Hello! How can I help you today?<|im_end|>
      ...

- The stop check in the serving code:

  ```python
  STOP = model.generation_config.eos_token_id
  ...
  if next_token == STOP:
      break
  ```

- The old model's `generation_config.json` has `"eos_token_id": 2`. The new
  one has `"eos_token_id": [151645, 151643]`.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 2: the new GPUs are slower

**Severity:** medium. **Reported by:** the platform team.

> We moved the private assistant from A100 to L40S. The L40S has more
> TFLOP/s, so we expected it to be faster. It is slower. Is the L40S driver
> broken?

**Evidence**

- The model is Llama-3-8B in bf16. Each GPU serves one user at a time, so the
  batch size is 1.
- On A100 the users got 104 tokens/s. On L40S they get 45 tokens/s.
- The team checked that the driver and the CUDA version are the ones that the
  vendor recommends.
- `nvidia-smi` on the L40S shows 99% "utilization" during generation.
- The prompts are short, less than 200 tokens. The replies are long, about 600
  tokens.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 3: the capacity plan

**Severity:** high. **Reported by:** the load test.

> The capacity plan says that one 24 GB card holds 85 concurrent requests. The
> load test gets out of memory at the 43rd request.

**Evidence**

- The model is Qwen3-0.6B. Each request can grow to 4096 tokens.
- The plan gives 20 GB to the KV cache. It keeps the rest for the weights, the
  activations and the CUDA context.
- The script of the plan:

  ```python
  head_dim = config.hidden_size // config.num_attention_heads
  kv_per_token = 2 * config.num_hidden_layers * config.num_key_value_heads * head_dim * 2
  per_request = kv_per_token * 4096
  print(20e9 // per_request)          # 85.0
  ```

- An extract from `config.json`:

  ```json
  "hidden_size": 1024,
  "num_attention_heads": 16,
  "num_key_value_heads": 8,
  "head_dim": 128,
  "num_hidden_layers": 28,
  ```

- Last week the team updated the CUDA driver.
- At the crash, the memory monitor shows 23.9 GB in use with 42 requests
  active.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 4: the kernel that was too fast

**Severity:** low. But it goes into a release note. **Reported by:** a code
reviewer.

> A pull request says: "the new attention kernel makes prefill 1.5x faster".
> Should we merge it?

**Evidence**

- The model is Qwen3-1.7B. The prompt has 2048 tokens. The card sustains 49
  TFLOP/s on a large bf16 matmul (`./vc info`).
- The benchmark in the PR:

  ```python
  model(prompt_ids)                 # warm-up
  start = time.perf_counter()
  model(prompt_ids)
  ms = (time.perf_counter() - start) * 1000
  ```

- The PR reports 123 ms with the new kernel. Main gives 188 ms with a
  benchmark that another team wrote.
- A second engineer repeats the PR benchmark with 40 iterations, and gets 183
  ms. They write: "the speedup goes away over a long run, so the card must
  throttle when it is hot."
- A forward pass costs about `2 x parameters x tokens` FLOP.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 5: the same question, a different answer

**Severity:** the customer says critical. **Reported by:** a customer.

> We use temperature 0. We send the same prompt two times and we get two
> different answers. Your service is not deterministic. Fix it.

**Evidence**

- The service runs Qwen3-1.7B in bf16 with a KV cache. It batches the requests
  of many users.
- The two answers are identical for 31 tokens. They differ from token 32 on.
- The team replays the two requests offline. At token 32, the top two logits
  are `17.125` and `17.000`.
- The two requests arrived at different times: one at 03:00, when the server
  was idle, and one at 14:00, when it was busy.
- The team runs both requests again in fp32. The two answers are identical.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 6: the bot ignores the question

**Severity:** high. **Reported by:** customer support.

> For long questions, the support bot writes an answer that has nothing to do
> with the question. For short questions it works.

**Evidence**

- Every request is the system prompt, then the question of the user. The
  system prompt has 480 tokens.
- The tokenization code:

  ```python
  ids = tokenizer(system_prompt + question, return_tensors='pt',
                  truncation=True, max_length=512).input_ids
  ```

  A developer copied it from a classification project last quarter.
- The histogram of the prompt lengths that the model sees has a tall spike at
  exactly 512 tokens.
- The model's context window is 40,960 tokens.
- The problem started after the system prompt grew from 200 to 480 tokens.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 7: the chatbot gets slower as it talks

**Severity:** medium. **Reported by:** users.

> The first words come fast. A long answer slows down until it crawls.

**Evidence**

- The model is Qwen3-1.7B on the same card as yours.
- The latency of each token, from the logs:

  | position of the token | ms for this token |
  |---|---|
  | 128 | 17 |
  | 1,024 | 95 |
  | 2,048 | 190 |

- The on-call engineer writes: "attention reads the whole context, so the
  step time must grow with the context. This is normal."
- The generation loop:

  ```python
  for _ in range(max_tokens):
      out = model(token_ids, use_cache=True)
      next_token = out.logits[0, -1].argmax()
      token_ids = torch.cat([token_ids, next_token.view(1, 1)], dim=1)
  ```

- A decode step at position 128 on this card, with a correct cache, takes
  about 13 ms.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 8: my answer has someone else's words in it

**Severity:** critical. It is a possible data leak. **Reported by:** a user.

> I asked for a pancake recipe. The answer started well. Then it talked about
> a cat and a dog with Japanese names. I never wrote about pets or Japan.

**Evidence**

- The first request after a restart is always correct.
- A debug log prints the length of the KV cache after each prefill:

  | request | prompt tokens | cache length after prefill |
  |---|---|---|
  | 1 | 8 | 8 |
  | 2 | 7 | 45 |
  | 3 | 12 | 97 |

- The request just before the pancake request asked about the largest cities
  in Japan.
- The generation module:

  ```python
  _cache = DynamicCache()           # module level

  def generate(model, tokenizer, prompt, max_tokens):
      out = model(ids, past_key_values=_cache, use_cache=True)
      ...
  ```

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 9: the long document

**Severity:** high. **Reported by:** the summarization team.

> Documents of 24,000 tokens crash with out-of-memory. Our plan says that
> they fit.

**Evidence**

- The model is Qwen3-1.7B on a 12 GB card, one request at a time.
- The plan: 3.44 GB of weights, plus 24,000 tokens of KV cache, plus 1.5 GB
  of activations and context. The plan says that this is less than 8 GB.
- The memory profiler shows one very large allocation, at the end of the
  prefill forward pass. It is 7.29 GB.
- The code of the prefill:

  ```python
  out = model(document_ids, use_cache=True)
  next_token = out.logits[0, -1].argmax()
  ```

- Documents of 8,000 tokens work.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 10: the first request of the day

**Severity:** medium. **Reported by:** the SRE team.

> After every deploy, and every time the autoscaler adds a pod, the first
> user on that pod gets a timeout. The users after it are fine.

**Evidence**

- The timeout for the first token is 500 ms.
- A prefill of 2048 tokens on a new pod: 644 ms for the first request, 186 ms
  for every request after it.
- The readiness probe calls `/health`. That route returns `200 OK` when the
  process has loaded the weights. It does not run the model.
- The autoscaler adds pods at the traffic peak, so the new pods get users at
  once.
- The weights load from a local disk in 4 s.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 11: we bought four times too many GPUs

**Severity:** medium. It costs money, not uptime. **Reported by:** the CFO.

> We planned 39 A100s for 500 concurrent users. The dashboard says that the
> KV cache is never more than 24% full, even at the peak. Why did we buy them?

**Evidence**

- The model is Llama-3-8B. Each user can reach 8192 tokens.
- On each 80 GB card, the plan gives 60 GB to the KV cache.
- The script of the plan:

  ```python
  kv_per_token = (2 * config.num_hidden_layers * config.num_attention_heads
                  * (config.hidden_size // config.num_attention_heads) * 2)
  users_per_gpu = int(60e9 // (kv_per_token * 8192))    # 13
  gpus = math.ceil(500 / users_per_gpu)                  # 39
  ```

- No request was ever rejected, and the latency is good.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

### Before you open the solution

Go back to each ticket and write one more line: **which piece of evidence was
noise, and why did it look relevant?** In real incidents the noise costs more
time than the bug.

Then do the second notebook of this section. In that notebook you make the
bugs yourself and watch them.